# MimicAI Custom Voice Fine-Tuner
This notebook trains a custom **Piper TTS** model on your voice dataset to capture both your **voice tone and way of speaking (prosody)**. 

**Hardware:** Ensure you are running this with a T4 GPU (Runtime > Change runtime type > T4 GPU).

## 1. Setup Environment

In [ ]:
!pip install numpy==1.24.4
!git clone https://github.com/rhasspy/piper.git
!cd piper/src/python && pip install -e .
!pip install pytorch-lightning==1.9.5
!apt-get install -y espeak-ng
!pip install tensorboard

## 2. Mount Google Drive
Upload the `dataset` folder (generated by `prepare_dataset.py`) to your Google Drive root directory.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/dataset"
if os.path.exists(dataset_path):
    print("Dataset found!")
else:
    print("Dataset NOT found! Please ensure 'dataset' folder is in your Google Drive root.")

## 3. Download Base Model
We will fine-tune from a high-quality base model (lessac) rather than training from scratch. This saves days of training.

In [ ]:
import urllib.request
os.makedirs("model", exist_ok=True)

print("Downloading base model checkpoint...")
ckpt_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/epoch%3D2159-step%3D1055532.ckpt"
urllib.request.urlretrieve(ckpt_url, "model/base.ckpt")
print("Downloaded successfully.")

## 4. Preprocess Dataset
Convert the raw audio and text into the binary format Piper expects.

In [ ]:
!python -m piper_train.preprocess \
  --language en-us \
  --dataset-dir /content/drive/MyDrive/dataset \
  --output-dir /content/training \
  --dataset-format ljspeech \
  --single-speaker

## 5. Train the Model
This will take about 1-2 hours depending on dataset size. You can stop it at any time, but 1000-2000 epochs is recommended.

In [ ]:
# Set epochs based on your dataset size
EPOCHS = 2000
BATCH_SIZE = 8

!python -m piper_train \
  --dataset-dir /content/training \
  --checkpoint-epochs 50 \
  --resume_from_checkpoint model/base.ckpt \
  --max_epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --quality medium

## 6. Export to ONNX
Convert the trained checkpoint into a lightweight `.onnx` model that works on your 4GB RAM local machine.

In [ ]:
import glob

# Find latest checkpoint
ckpts = glob.glob("/content/training/lightning_logs/version_0/checkpoints/*.ckpt")
latest_ckpt = max(ckpts, key=os.path.getctime)
print(f"Exporting checkpoint: {latest_ckpt}")

!python -m piper_train.export_onnx \
  {latest_ckpt} \
  /content/drive/MyDrive/my_custom_voice.onnx
  
print("==========================================")
print("DONE! Download 'my_custom_voice.onnx' from your Google Drive and place it in the 'piper_models' folder in MimicAI.")